In [4]:
# -*- coding: utf-8 -*-
"""
Reddit Sampling Pipeline
--------------------------------------------------------
Requires the data to be stored in a folder called 'data'.
Used on the subreddits 

r/changemyview
r/AskReddit
r/TrueReddit
r/AmItheAsshole
r/debate
r/PoliticalDiscussion
r/liberal
r/conservative
r/moderatepolitics
r/worldnews
r/socialjustice


This pipeline only looks at the posts itself, the comments have to be matched in a second step. 
threadminer
"""

"\nReddit Sampling Pipeline\n--------------------------------------------------------\nRequires the data to be stored in a folder called 'data'.\nUsed on the subreddits \n\nr/changemyview\nr/AskReddit\nr/TrueReddit\nr/AmItheAsshole\nr/debate\nr/PoliticalDiscussion\nr/liberal\nr/conservative\nr/moderatepolitics\nr/worldnews\nr/socialjustice\n\n\nThis pipeline only looks at the posts itself, the comments have to be matched in a second step. \nthreadminer\n"

In [5]:
from __future__ import annotations
import hashlib
from typing import Iterable, List, Optional, Tuple, Dict, Union
import numpy as np
import pandas as pd
from pathlib import Path
from glob import glob

In [7]:
import json
from pathlib import Path
from typing import Union, Iterable, List, Tuple
import pandas as pd
from glob import glob


# Columns to retain from the original Reddit JSONL files
KEEP_COLS = ["author", "created_utc", "downs", "id", "likes", "num_comments", "ups", "selftext", "title", "subreddit"]

def load_reddit_jsonl_files(
        paths: Union[str, Path, Iterable[Union[str, Path]]],
        *,
        verbose: bool = True,
        max_error_previews: int = 3,
) -> pd.DataFrame:
    """
    Robust loader for one or multiple Reddit JSONL files.

    Functionality:
    - Reads JSON Lines (.jsonl) files line-by-line (robust to malformed JSON rows)
    - Keeps only predefined columns
    - Converts `created_utc` (Unix timestamp) into UTC datetime
    - Ensures consistent schema across all input files
    - Logs skipped malformed lines per file
    """

    # Convert a single path into a list
    if isinstance(paths, (str, Path)):
        paths = [paths]

    dfs: List[pd.DataFrame] = []
    total_skipped = 0
    processed_files = 0
    failed_files = 0

    for p in paths:
        p = Path(p)
        if not p.exists():
            if verbose:
                print(f"[WARN] File not found and skipped: {p}")
            continue

        rows = []
        skipped = 0
        previews: List[Tuple[int, str]] = []

        try:
            with p.open("r", encoding="utf-8") as f:
                for i, line in enumerate(f, 1):
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        rows.append(json.loads(line))
                    except Exception as e:
                        skipped += 1
                        if len(previews) < max_error_previews:
                            previews.append((i, str(e)))

            processed_files += 1
            total_skipped += skipped

            if verbose and skipped > 0:
                print(f"[WARN] {p.name}: skipped {skipped} malformed lines")
                for ln, msg in previews:
                    print(f"       - line {ln}: {msg}")

            if not rows:
                # Entire file was empty or fully malformed
                if verbose:
                    print(f"[WARN] {p.name}: no valid rows found (empty or fully malformed)")
                continue

            df = pd.DataFrame(rows)

            # Ensure missing expected columns exist (filled with NA)
            for c in KEEP_COLS:
                if c not in df.columns:
                    df[c] = pd.NA

            # Select only the relevant columns
            df = df[KEEP_COLS].copy()

            # Convert Unix timestamp to UTC datetime (robust to bad types)
            # Some datasets store created_utc as int/float seconds; others may contain strings.
            df["created_utc"] = pd.to_numeric(df["created_utc"], errors="coerce")
            df["created_utc"] = pd.to_datetime(df["created_utc"], unit="s", utc=True, errors="coerce")

            dfs.append(df)

        except Exception as e:
            failed_files += 1
            if verbose:
                print(f"[ERROR] Failed to read {p.name}: {e}")
            continue

    # If no file was successfully processed, return an empty DataFrame with the correct schema
    if not dfs:
        empty_cols = ["id", "author", "created_utc", "ups", "downs", "likes", "num_comments", "selftext", "title", "subreddit"]
        if verbose:
            print("[INFO] No data loaded. Returning empty DataFrame.")
        return pd.DataFrame(columns=empty_cols)

    # Concatenate all loaded DataFrames
    df_all = pd.concat(dfs, ignore_index=True)

    # Order columns in a logical sequence
    out_cols = ["id", "author", "created_utc", "ups", "downs", "likes", "num_comments", "selftext", "title", "subreddit"]
    # Ensure output columns exist (defensive)
    for c in out_cols:
        if c not in df_all.columns:
            df_all[c] = pd.NA
    df_all = df_all[out_cols]

    if verbose:
        print("\n--- Load summary ---")
        print("Processed files:", processed_files)
        print("Failed files:", failed_files)
        print("Total skipped malformed lines:", total_skipped)
        print("Total rows loaded:", len(df_all))

    return df_all


# ==========================================
# usage
# ==========================================

files = glob("data/*_posts.jsonl")
df_all = load_reddit_jsonl_files(files, verbose=True)

print(df_all.head())
print("Rows:", len(df_all))



[WARN] r_politics_posts.jsonl: skipped 1 malformed lines
       - line 570: Unterminated string starting at: line 1 column 689 (char 688)

--- Load summary ---
Processed files: 12
Failed files: 0
Total skipped malformed lines: 1
Total rows loaded: 174899
        id                author               created_utc  ups  downs likes  \
0  1dc82va           clonedhuman 2024-06-10 00:00:20+00:00    1      0  None   
1  1dc893q  nosotros_road_sodium 2024-06-10 00:08:30+00:00   52      0  None   
2  1dc8bmy          Shigpossposs 2024-06-10 00:11:58+00:00    1      0  None   
3  1dc8g7e  nosotros_road_sodium 2024-06-10 00:18:27+00:00    0      0  None   
4  1dc8i3l  Ecstatic-Medium-6320 2024-06-10 00:21:17+00:00    2      0  None   

   num_comments   selftext                                              title  \
0             2             Republicans Try To Block 4 Million Workers Fro...   
1            38             US, Saudi Arabia close to finalizing draft sec...   
2             1  [rem

In [9]:
# Optional NLP dependencies
_HAS_SBERT = False
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity
    _HAS_SBERT = True
except Exception:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity


# ------------------------
# Topics
# ------------------------

DEFAULT_TOPICS = [
    "The government should not forgive student loan debt.",
    "Airbnb should be banned in cities.",
    "The federal minimum wage should be increased.",
    "The US should provide financial and military aid to Ukraine.",
    "A universal basic income would kill the economy.",
    "Climate change is one of the greatest threats to humanity.",
    "Fur clothing should be banned.",
    "The government should not invest in renewable energy.",
    "There should only be vegetarian food in cantines.",
    "Gender-neutral language and stating pronouns are silly issues.",
    "Prostitution should be illegal.",
    "Employers should mandate vaccination.",
    "The government should not be responsible for universal health care.",
    "Immigrants should adopt the local language and culture.",
    "We need stricter gun control laws.",
    "The death penalty should be reestablished.",
    "Police officers should wear body cameras.",
    "Artificial Intelligence should replace humans where possible.",
    "Social media is a threat to democracy.",
]


# ------------------------
# Build topic model
# ------------------------

def build_topic_model(topics: List[str]):
    """
    Build topic embeddings (SBERT) or TF-IDF vectors.
    """
    if _HAS_SBERT:
        model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        topic_embeddings = model.encode(topics, show_progress_bar=False)
        return "sbert", model, topic_embeddings
    else:
        vec = TfidfVectorizer(min_df=1, max_df=0.95, ngram_range=(1, 2))
        topic_matrix = vec.fit_transform(topics)
        return "tfidf", vec, topic_matrix


# ------------------------
# Core filtering function
# ------------------------

def filter_by_topic_similarity(
        df: pd.DataFrame,
        similarity_threshold: float = 0.4,
        topics: Optional[List[str]] = None,
) -> Tuple[pd.DataFrame, int, int]:
    """
    Filter dataframe rows based on cosine similarity between `title`
    and a set of topic sentences.

    Requirements:
    - `df` must contain a column `title`.
    - Rows with title == "[removed]" or "[deleted]" are discarded immediately.

    Returns:
    - filtered_df: rows with similarity >= threshold
    - kept: number of rows retained
    - dropped: number of rows removed
    """

    if "title" not in df.columns:
        raise ValueError("Expected column 'title' in dataframe.")

    # Work on a copy
    df = df.copy()

    # Remove [removed] / [deleted] immediately
    mask_valid = ~df["title"].isin(["[removed]", "[deleted]"])
    df = df[mask_valid].copy()
    

    if df.empty:
        return df, 0, 0

    topics = topics or DEFAULT_TOPICS
    texts = df["title"].fillna("").astype(str).tolist()

    # Build topic model
    kind, model, topic_repr = build_topic_model(topics)

    # Compute similarities
    if kind == "sbert":
        text_embeddings = model.encode(texts, show_progress_bar=False)
        sims = cosine_similarity(text_embeddings, topic_repr)

    else:
        # TF-IDF fallback: fit on topics + texts
        vec = TfidfVectorizer(min_df=1, max_df=0.95, ngram_range=(1, 2))
        combined = topics + texts
        X = vec.fit_transform(combined)
        topic_matrix = X[: len(topics), :]
        text_matrix = X[len(topics):, :]
        sims = cosine_similarity(text_matrix, topic_matrix)

    # Determine best topic per row
    best_idx = sims.argmax(axis=1)
    best_sim = sims.max(axis=1)

    df["best_topic_index"] = best_idx
    df["best_topic"] = [topics[i] for i in best_idx]
    df["best_topic_similarity"] = best_sim

    # Threshold filtering
    keep_mask = df["best_topic_similarity"] >= similarity_threshold
    filtered_df = df[keep_mask].copy()

    kept = int(keep_mask.sum())
    dropped = int(len(df) - kept)

    print(f"Submissions kept: {kept}")
    print(f"Submissions dropped: {dropped}")
    
    return filtered_df, kept, dropped


# ------------------------
# Example usage
# ------------------------

if __name__ == "__main__":
   
    df_posts = df_all
    print(len(df_posts))

    SIMILARITY_THRESHOLD = 0.4
    filtered, kept, dropped = filter_by_topic_similarity(
        df=df_posts,             # your main dataframe from loader
        similarity_threshold=SIMILARITY_THRESHOLD,
        topics=DEFAULT_TOPICS,
    )

    print(filtered.head())
    print(f"Final kept: {kept}, dropped: {dropped}")


174899
Submissions kept: 3125
Submissions dropped: 171774
          id                author               created_utc   ups  downs  \
0    1dc82va           clonedhuman 2024-06-10 00:00:20+00:00     1      0   
84   1dcetz6         Kate_Matthews 2024-06-10 06:19:49+00:00  1071      0   
108  1dch1vs  LbextDipsBenchPlatty 2024-06-10 09:01:08+00:00     1      0   
129  1dcjmbw       eustachian_lube 2024-06-10 11:51:54+00:00     0      0   
160  1dcngsr         Ok-Story-9319 2024-06-10 14:54:58+00:00     1      0   

    likes  num_comments   selftext  \
0    None             2              
84   None            96              
108  None             1              
129  None            43              
160  None             1  [removed]   

                                                 title subreddit  \
0    Republicans Try To Block 4 Million Workers Fro...  politics   
84   US pushes for $50 billion loan to Ukraine usin...  politics   
108  The risks of AI could be catastrophic. We

In [10]:
filtered.to_json("sampled.ndjson", orient="records", lines=True, force_ascii=False)